# 08 — XGBoost Model Training with Hyperparameter Tuning

**Owner:** Member 2 — Narendra Iyer 
**Project:** AeroDelay AI — Airline Delay Risk Prediction 
**Course:** AAI-540, Group 3

## Purpose
Train the main XGBoost classifier using a SageMaker Training Job, then run
Automatic Model Tuning (HPO) to find the best hyperparameter combination.
The EDA in notebook 03 showed a 4:1 class imbalance, so `scale_pos_weight=4`
is our starting point.

## Prerequisites
- `07_baseline_model.ipynb` completed — `%store` variables `baseline_job_name`
  and `baseline_model_uri` must be present.

## 1. Setup

In [1]:
import boto3, os, json, time
import pandas as pd
import awswrangler as wr
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess   = Session()
bucket = sess.default_bucket()
role   = get_execution_role()
region = sess.boto_region_name
sm     = boto3.client("sagemaker")

print(f"Bucket : {bucket}")
print(f"Region : {region}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Bucket : sagemaker-us-east-1-151132426745
Region : us-east-1


In [2]:
%store -r s3_aerodelay
%store -r baseline_job_name
%store -r baseline_model_uri
print(f"s3_aerodelay       : {s3_aerodelay}")
print(f"baseline_job_name  : {baseline_job_name}")
print(f"baseline_model_uri : {baseline_model_uri}")

s3_aerodelay       : s3://sagemaker-us-east-1-151132426745/airline-delay
baseline_job_name  : aerodelay-baseline-2026-06-13-17-38-24-100
baseline_model_uri : s3://sagemaker-us-east-1-151132426745/airline-delay/model-artifacts/baseline/aerodelay-baseline-2026-06-13-17-38-24-100/output/model.tar.gz


## 2. Write XGBoost Training Script

In [3]:
os.makedirs("src/training", exist_ok=True)

xgb_script = '''
# train_xgboost.py — XGBoost global model for AeroDelay AI
# Runs inside a SageMaker Training Job container.
import argparse, os, json
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

TARGET = "arrdel15"

def load_parquet(path):
    files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(".parquet")]
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

def evaluate(model, dmatrix, y, split_name):
    y_prob = model.predict(dmatrix)
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "split"     : split_name,
        "accuracy"  : round(float(accuracy_score(y, y_pred)), 4),
        "precision" : round(float(precision_score(y, y_pred, zero_division=0)), 4),
        "recall"    : round(float(recall_score(y, y_pred, zero_division=0)), 4),
        "f1"        : round(float(f1_score(y, y_pred, zero_division=0)), 4),
        "roc_auc"   : round(float(roc_auc_score(y, y_prob)), 4),
    }
    cm = confusion_matrix(y, y_pred).tolist()
    print(json.dumps({"metrics": metrics, "confusion_matrix": cm}))
    return metrics, cm

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train",            default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--validation",       default=os.environ.get("SM_CHANNEL_VALIDATION"))
    parser.add_argument("--model-dir",        default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--output-data-dir",  default=os.environ.get("SM_OUTPUT_DATA_DIR"))
    parser.add_argument("--num-round",        type=int,   default=200)
    parser.add_argument("--max-depth",        type=int,   default=6)
    parser.add_argument("--eta",              type=float, default=0.1)
    parser.add_argument("--subsample",        type=float, default=0.8)
    parser.add_argument("--colsample-bytree", type=float, default=0.8)
    parser.add_argument("--min-child-weight", type=int,   default=5)
    parser.add_argument("--scale-pos-weight", type=float, default=4.0)  # 4:1 imbalance from EDA
    args = parser.parse_args()

    print("Loading data...")
    train_df = load_parquet(args.train)
    val_df   = load_parquet(args.validation)

    feature_cols = [c for c in train_df.columns if c != TARGET]
    X_train = train_df[feature_cols].fillna(0).values
    y_train = train_df[TARGET].values
    X_val   = val_df[feature_cols].fillna(0).values
    y_val   = val_df[TARGET].values

    print(f"Train : {X_train.shape} | delay rate {y_train.mean():.3f}")
    print(f"Val   : {X_val.shape}   | delay rate {y_val.mean():.3f}")

    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_cols)
    dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=feature_cols)

    params = {
        "objective"        : "binary:logistic",
        "eval_metric"      : ["auc", "logloss"],
        "max_depth"        : args.max_depth,
        "eta"              : args.eta,
        "subsample"        : args.subsample,
        "colsample_bytree" : args.colsample_bytree,
        "min_child_weight" : args.min_child_weight,
        "scale_pos_weight" : args.scale_pos_weight,
        "seed"             : 42,
    }

    print("Training XGBoost...")
    evals_result = {}
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=args.num_round,
        evals=[(dtrain, "train"), (dval, "validation")],
        early_stopping_rounds=20,
        evals_result=evals_result,
        verbose_eval=50
    )

    best_iter = model.best_iteration
    val_auc   = evals_result["validation"]["auc"][best_iter]
    # SageMaker HPO reads this exact line to extract the objective metric
    print(f"validation:auc={val_auc}")
    print(f"Best iteration: {best_iter}")

    train_metrics, train_cm = evaluate(model, dtrain, y_train, "train")
    val_metrics,   val_cm   = evaluate(model, dval,   y_val,   "validation")

    # Feature importance (top 15 by gain)
    importance = model.get_score(importance_type="gain")
    top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:15]
    print("Top 15 features by gain:")
    for feat, score in top_features:
        print(f"  {feat}: {score:.2f}")

    os.makedirs(args.output_data_dir, exist_ok=True)
    output = {
        "model"           : "XGBoost",
        "best_iteration"  : best_iter,
        "train"           : {"metrics": train_metrics, "confusion_matrix": train_cm},
        "validation"      : {"metrics": val_metrics,   "confusion_matrix": val_cm},
        "top_features"    : dict(top_features),
        "hyperparameters" : vars(args),
    }
    with open(os.path.join(args.output_data_dir, "xgb_metrics.json"), "w") as fp:
        json.dump(output, fp, indent=2)

    model.save_model(os.path.join(args.model_dir, "xgboost-model"))
    joblib.dump(feature_cols, os.path.join(args.model_dir, "feature_cols.joblib"))
    print("Model and feature list saved.")
'''

with open("src/training/train_xgboost.py", "w") as f:
    f.write(xgb_script.strip())

print("Written: src/training/train_xgboost.py")

Written: src/training/train_xgboost.py


## 3. Run Initial XGBoost Training Job (Default Hyperparameters)

From the EDA findings:
- Class imbalance is 4:1 → `scale_pos_weight=4`  
- We prioritize recall first, then F1 → tree model is a good fit  
- XGBoost does not need StandardScaler (unlike the LR baseline)

In [13]:
os.makedirs("src/training", exist_ok=True)

xgb_script = '''
import argparse, os, json
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

TARGET = "arrdel15"

def load_parquet(path):
    files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(".parquet")]
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

def encode_strings(df):
    for col in df.select_dtypes(include=["object", "string"]).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    return df

def evaluate(model, dmatrix, y, split_name):
    y_prob = model.predict(dmatrix)
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "split"     : split_name,
        "accuracy"  : round(float(accuracy_score(y, y_pred)), 4),
        "precision" : round(float(precision_score(y, y_pred, zero_division=0)), 4),
        "recall"    : round(float(recall_score(y, y_pred, zero_division=0)), 4),
        "f1"        : round(float(f1_score(y, y_pred, zero_division=0)), 4),
        "roc_auc"   : round(float(roc_auc_score(y, y_prob)), 4),
    }
    cm = confusion_matrix(y, y_pred).tolist()
    print(json.dumps({"metrics": metrics, "confusion_matrix": cm}))
    return metrics, cm

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train",            default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--validation",       default=os.environ.get("SM_CHANNEL_VALIDATION"))
    parser.add_argument("--model_dir",        default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--output_data_dir",  default=os.environ.get("SM_OUTPUT_DATA_DIR"))
    parser.add_argument("--num_round",        type=int,   default=200)
    parser.add_argument("--max_depth",        type=int,   default=6)
    parser.add_argument("--eta",              type=float, default=0.1)
    parser.add_argument("--subsample",        type=float, default=0.8)
    parser.add_argument("--colsample_bytree", type=float, default=0.8)
    parser.add_argument("--min_child_weight", type=int,   default=5)
    parser.add_argument("--scale_pos_weight", type=float, default=4.0)
    args = parser.parse_args()

    print("Loading data...")
    train_df = load_parquet(args.train)
    val_df   = load_parquet(args.validation)

    feature_cols = [c for c in train_df.columns if c != TARGET]
    X_train = encode_strings(train_df[feature_cols].fillna(0)).values
    y_train = train_df[TARGET].values
    X_val   = encode_strings(val_df[feature_cols].fillna(0)).values
    y_val   = val_df[TARGET].values

    print(f"Train : {X_train.shape} | delay rate {y_train.mean():.3f}")
    print(f"Val   : {X_val.shape}   | delay rate {y_val.mean():.3f}")

    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_cols)
    dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=feature_cols)

    params = {
        "objective"        : "binary:logistic",
        "eval_metric"      : ["auc", "logloss"],
        "max_depth"        : args.max_depth,
        "eta"              : args.eta,
        "subsample"        : args.subsample,
        "colsample_bytree" : args.colsample_bytree,
        "min_child_weight" : args.min_child_weight,
        "scale_pos_weight" : args.scale_pos_weight,
        "seed"             : 42,
    }

    print("Training XGBoost...")
    evals_result = {}
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=args.num_round,
        evals=[(dtrain, "train"), (dval, "validation")],
        early_stopping_rounds=20,
        evals_result=evals_result,
        verbose_eval=50
    )

    best_iter = model.best_iteration
    val_auc   = evals_result["validation"]["auc"][best_iter]
    print(f"validation:auc={val_auc}")
    print(f"Best iteration: {best_iter}")

    train_metrics, train_cm = evaluate(model, dtrain, y_train, "train")
    val_metrics,   val_cm   = evaluate(model, dval,   y_val,   "validation")

    importance = model.get_score(importance_type="gain")
    top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:15]
    print("Top 15 features by gain:")
    for feat, score in top_features:
        print(f"  {feat}: {score:.2f}")

    os.makedirs(args.output_data_dir, exist_ok=True)
    output = {
        "model"           : "XGBoost",
        "best_iteration"  : best_iter,
        "train"           : {"metrics": train_metrics, "confusion_matrix": train_cm},
        "validation"      : {"metrics": val_metrics,   "confusion_matrix": val_cm},
        "top_features"    : dict(top_features),
        "hyperparameters" : vars(args),
    }
    with open(os.path.join(args.output_data_dir, "xgb_metrics.json"), "w") as fp:
        json.dump(output, fp, indent=2)

    model.save_model(os.path.join(args.model_dir, "xgboost-model"))
    joblib.dump(feature_cols, os.path.join(args.model_dir, "feature_cols.joblib"))
    print("Model and feature list saved.")
'''

with open("src/training/train_xgboost.py", "w") as f:
    f.write(xgb_script.strip())

print("Written: src/training/train_xgboost.py")

Written: src/training/train_xgboost.py


## 4. Hyperparameter Tuning Job (HPO)

We optimize for `validation:auc` using 10 trials.  
`max_parallel_jobs=2` keeps cost low on the learner lab account.

In [14]:
from sagemaker.tuner import (
    HyperparameterTuner, IntegerParameter,
    ContinuousParameter
)

hpo_estimator = XGBoost(
    entry_point       = "train_xgboost.py",
    source_dir        = "src/training",
    framework_version = "1.7-1",
    instance_type     = "ml.m5.xlarge",
    instance_count    = 1,
    role              = standard_role,
    sagemaker_session = standard_sess,
    base_job_name     = "aerodelay-hpo",
    hyperparameters   = {"scale_pos_weight": 4.0},
    output_path       = f"{s3_aerodelay}/model-artifacts/hpo/",
)

hyperparameter_ranges = {
    "max_depth"        : IntegerParameter(3, 9),
    "eta"              : ContinuousParameter(0.01, 0.3),
    "subsample"        : ContinuousParameter(0.5, 1.0),
    "colsample_bytree" : ContinuousParameter(0.5, 1.0),
    "min_child_weight" : IntegerParameter(1, 10),
    "num_round"        : IntegerParameter(100, 400),
}

tuner = HyperparameterTuner(
    estimator             = hpo_estimator,
    objective_metric_name = "validation:auc",
    objective_type        = "Maximize",
    hyperparameter_ranges = hyperparameter_ranges,
    max_jobs              = 10,
    max_parallel_jobs     = 2,
    base_tuning_job_name  = "aerodelay-tuning",
)

tuner.fit(
    inputs={
        "train"      : f"{s3_aerodelay}/training/",
        "validation" : f"{s3_aerodelay}/validation/",
    },
    wait=True,
    logs=False
)

print("\nHPO complete.")

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.xlarge.


INFO:sagemaker:Creating hyperparameter tuning job with name: aerodelay-tuning-260613-1814


.

.

...

.

.

.

!



HPO complete.


## 5. Get Best Hyperparameters

In [15]:
best_hpo_job_name = tuner.best_training_job()
best_job_desc     = sm.describe_training_job(TrainingJobName=best_hpo_job_name)
best_hp           = best_job_desc["HyperParameters"]
best_model_uri    = best_job_desc["ModelArtifacts"]["S3ModelArtifacts"]
hpo_job_name      = tuner.latest_tuning_job.name
xgb_initial_job   = xgb_estimator.latest_training_job.name

print("Best HPO job   :", best_hpo_job_name)
print("Best model URI :", best_model_uri)
print("Best hyperparameters:")
for k, v in sorted(best_hp.items()):
    print(f"  {k}: {v}")

Best HPO job   : aerodelay-tuning-260613-1814-001-9a5121f3
Best model URI : s3://sagemaker-us-east-1-151132426745/airline-delay/model-artifacts/hpo/aerodelay-tuning-260613-1814-001-9a5121f3/output/model.tar.gz
Best hyperparameters:
  _tuning_objective_metric: validation:auc
  colsample_bytree: 0.5481133017459849
  eta: 0.1508464235336603
  max_depth: 9
  min_child_weight: 8
  num_round: 262
  sagemaker_container_log_level: 20
  sagemaker_estimator_class_name: "XGBoost"
  sagemaker_estimator_module: "sagemaker.xgboost.estimator"
  sagemaker_job_name: "aerodelay-hpo-2026-06-13-18-14-39-553"
  sagemaker_program: "train_xgboost.py"
  sagemaker_region: "us-east-1"
  sagemaker_submit_directory: "s3://sagemaker-us-east-1-151132426745/aerodelay-hpo-2026-06-13-18-14-39-553/source/sourcedir.tar.gz"
  scale_pos_weight: 4.0
  subsample: 0.8798494705937904


## 6. Store Job Details for Downstream Notebooks

In [16]:
%store xgb_initial_job
%store hpo_job_name
%store best_hpo_job_name
%store best_model_uri
%store best_hp

print("Stored: xgb_initial_job, hpo_job_name, best_hpo_job_name, best_model_uri, best_hp")

Stored 'xgb_initial_job' (str)
Stored 'hpo_job_name' (str)
Stored 'best_hpo_job_name' (str)
Stored 'best_model_uri' (str)
Stored 'best_hp' (dict)
Stored: xgb_initial_job, hpo_job_name, best_hpo_job_name, best_model_uri, best_hp


## 7. Summary - XGBoost Training and HPO Results

In this notebook I trained the main XGBoost classifier and ran Automatic Model Tuning (HPO) across 10 trials to find the best hyperparameter combination. The objective metric was `validation:auc`, and I used 2 parallel jobs to keep costs manageable within the Learner Lab environment.

**HPO Job:** `aerodelay-tuning-260613-1814`
- Total trials: 10
- Parallel jobs: 2
- Objective: Maximize `validation:auc`

**Best Trial:** `aerodelay-tuning-260613-1814-001-9a5121f3`

**Best Hyperparameters Discovered:**

| Hyperparameter | Value | Notes |
|---|---|---|
| max_depth | 9 | Deep trees — the model found complex interactions worth capturing |
| eta (learning rate) | 0.151 | Moderate learning rate, balanced with 262 rounds |
| num_round | 262 | Sufficient boosting rounds without overfitting |
| colsample_bytree | 0.548 | ~55% of features sampled per tree — good regularization |
| subsample | 0.880 | 88% of training rows used per tree |
| min_child_weight | 8 | Higher value reduces overfitting on minority class |
| scale_pos_weight | 4.0 | Fixed — reflects the 4:1 class imbalance from EDA |

**Key Observations:**
The HPO found that deeper trees (max_depth=9) with a moderate learning rate and a relatively high number of rounds performed best. The `colsample_bytree` of 0.548 means each tree only sees about half the features, which adds diversity and reduces variance. The `min_child_weight=8` provides additional protection against overfitting on the minority delay class.

All job details — initial job name, HPO job name, best trial name, best model URI, and best hyperparameters — are stored via `%store` for use in notebooks 09 and 10.